**Import Libraries**

In [1]:
import os
import tarfile
from collections import Counter
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split


**Extract the Folders Bec each Celebrity has its Own Folder**

In [3]:
tgz_path = '/content/lfw-funneled.tgz'
extract_path = '/content/lfw_images'

with tarfile.open(tgz_path, 'r:gz') as tar:
    tar.extractall(path=extract_path)

print("Extraction complete!")
print(os.listdir(extract_path))


/tmp/ipython-input-289050390.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Extraction complete!
['lfw_funneled']


**Check for the First 10 Classes of images of Celebrities Bec each one has its own Class**

In [4]:
dataset_path = '/content/lfw_images/lfw_funneled'
print(os.listdir(dataset_path)[:10])


['Dimitar_Berbatov', 'Mary_Katherine_Smart', 'Juan_Carlos_Morales', 'Julio_De_Brun', 'Liam_Neeson', 'Sheldon_Silver', 'Jimmy_Iovine', 'Patrick_Stewart', 'Ellen_DeGeneres', 'Lauren_Hutton']


**Define Transforms for Each Image To Standard Each Image in the Model**

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


**Load Dataset and Check for the Total images and number of Classes of Celebrities and we read the Path Correctly**

In [6]:
full_dataset = datasets.ImageFolder(root=dataset_path, transform=train_transform)
print("Total images:", len(full_dataset))
print("Total classes:", len(full_dataset.classes))


Total images: 13233
Total classes: 5749


# Keeps only the Folder that have a Celebritie that has 5 images or more to Train the Model well and having a Good Accuracy

In [12]:
labels=[label for _, label in full_dataset.samples]
label_count=Counter(labels)
Min_Images=5
valid_labels={label for label,count in label_count.items() if count >= Min_Images}
filtered_indices=[
    i for i, (_, label) in enumerate(full_dataset.samples)
    if label in valid_labels
]
filtered_dataset=Subset(full_dataset,filtered_indices)
print("Total images:",len(filtered_dataset))
print("the Filtered Classes are:",valid_labels)
print("Total Classes:",len(valid_labels))


Total images: 5985
the Filtered Classes are: {2055, 20, 4116, 4119, 2085, 2088, 52, 60, 4161, 67, 4167, 2131, 84, 87, 4184, 4187, 93, 2141, 2145, 103, 2154, 107, 2155, 2163, 2171, 127, 2175, 4242, 4243, 2199, 2210, 4268, 4269, 4274, 182, 4290, 4296, 204, 4304, 210, 4308, 2265, 222, 223, 2276, 2279, 4327, 237, 4333, 239, 2288, 2290, 248, 2298, 2307, 4371, 2332, 291, 299, 302, 304, 4403, 2361, 317, 321, 4424, 2381, 4436, 354, 359, 4456, 4459, 370, 373, 2424, 380, 385, 387, 2438, 4487, 2444, 4492, 401, 2454, 2462, 417, 2467, 2468, 2477, 4541, 2499, 4551, 2506, 2507, 2510, 2514, 4566, 472, 4572, 2526, 4577, 2530, 4581, 487, 495, 4592, 4593, 502, 507, 2556, 4606, 2570, 4622, 531, 537, 538, 539, 2585, 4633, 549, 560, 4657, 2614, 570, 2623, 4672, 580, 4676, 2630, 4680, 4692, 600, 4699, 606, 4704, 2659, 615, 4716, 629, 2679, 2680, 2682, 2688, 641, 2705, 4760, 4766, 2721, 2725, 4773, 2735, 4784, 2738, 4786, 4789, 2746, 2750, 706, 2768, 2772, 4825, 2779, 736, 4833, 4836, 744, 747, 4845, 2801, 28


# ** Split The Filtered Dataset into Train,Test Validation**

In [13]:
train_size=int(0.8 * len(filtered_dataset))
test_size=len(filtered_dataset) - train_size
train_dataset,test_dataset=random_split(
    filtered_dataset, [train_size,test_size]
)
test_dataset.dataset.transform=val_transform

In [14]:
print(len(train_dataset))
print(len(test_dataset))


4788
1197


# Create DataLoaders , Batches and implement Shuffle  only on train_dataset

In [15]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)